In [1]:
'''
notebook for extracting variables from GLODAPv2.2023_Merged_Master_File.csv files and saving on disk

author: Parsa Gooya (parsa.gooya@ec.gc.ca)
'''

'\nnotebook for extracting variables from GLODAPv2.2023_Merged_Master_File.csv files and saving on disk\n\nauthor: Parsa Gooya (parsa.gooya@ec.gc.ca)\n'

In [2]:
import warnings
warnings.filterwarnings('ignore') # don't output warnings
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from pathlib import Path
import pandas as pd 

========================================================================================================================================================

User configs:

In [ ]:
glodap_dir = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data/GLODAP/GLODAPv2.2023_Merged_Master_File.csv'
output_dir = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data/'

var_list = ['silicate', 'phosphate', 'nitrite', 'oxygen','talk','theta', 'salinity', 'tco2']
#var_list =   [ 'nitrate', 'phosphate', 'silicate']


========================================================================================================================================================

In [8]:
data = pd.read_csv(glodap_dir)

In [9]:
display(data)

,G2expocode,G2cruise,G2station,G2region,G2cast,G2year,G2month,G2day,G2hour,G2minute,...,G2tocf,G2doc,G2docf,G2don,G2donf,G2tdn,G2tdnf,G2chla,G2chlaf,G2doi
0,06AQ19840719,1.0,319.0,4.0,1.0,1984.0,7.0,20.0,14.0,46.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.3334/cdiac/otg.carina_06aq1...
1,06AQ19840719,1.0,319.0,4.0,1.0,1984.0,7.0,20.0,14.0,46.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.3334/cdiac/otg.carina_06aq1...
2,06AQ19840719,1.0,319.0,4.0,1.0,1984.0,7.0,20.0,14.0,46.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.3334/cdiac/otg.carina_06aq1...
3,06AQ19840719,1.0,319.0,4.0,1.0,1984.0,7.0,20.0,14.0,46.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.3334/cdiac/otg.carina_06aq1...
4,06AQ19840719,1.0,319.0,4.0,1.0,1984.0,7.0,20.0,14.0,46.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.3334/cdiac/otg.carina_06aq1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402824,77DN20210725,5023.0,58.0,4.0,18.0,2021.0,9.0,11.0,13.0,27.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.25921/eaf4-9658
1402825,77DN20210725,5023.0,58.0,4.0,18.0,2021.0,9.0,11.0,13.0,27.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.25921/eaf4-9658
1402826,77DN20210725,5023.0,58.0,4.0,18.0,2021.0,9.0,11.0,13.0,27.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.25921/eaf4-9658
1402827,77DN20210725,5023.0,58.0,4.0,18.0,2021.0,9.0,11.0,13.0,27.0,...,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,-9999.0,9.0,https://doi.org/10.25921/eaf4-9658


In [ ]:
obs_to_model_rename = {'salinity' : 'so' , 'theta' : 'thetao', 'oxygen' :'o2', 'tco2' : 'dissic', 'nitrate' : 'no3', 'phosphate' : 'po4'}
def csv_var(data, var):
    
    columns_to_extract = ['G2year', 'G2month', 'G2day', 'G2hour',  'G2minute', 'G2latitude', 'G2longitude', 'G2depth', 'G2pressure', f'G2{var}']  
    var_data = data[columns_to_extract]
    rename_dict = {}
    for item in columns_to_extract:
        rename_dict[item] =  item.split('G2')[1]

    var_data.rename(columns=rename_dict, inplace=True)
    
    var_data[var] = var_data[var].replace(-9999, np.nan)
    var_data_columns = list(var_data.columns)
    var_data['date'] = pd.to_datetime(var_data[['year', 'month', 'day', 'hour', 'minute']])
    var_data = var_data[['date', *var_data_columns]]
    var_data.rename(columns={'month' : 'time', 'latitude' : 'lat', 'longitude' : 'lon', 'depth' : 'lev'}, inplace=True)
    var_data.rename(columns=obs_to_model_rename, inplace=True)
    return var_data



In [ ]:
for var in var_list:


    var_data = csv_var(data, var)
    try:
        var = obs_to_model_rename[var]
    except:
        pass
    dirout = f'{output_dir}/{var}/observation'
    Path(dirout).mkdir(parents=True, exist_ok=True)
    var_data.to_csv(dirout + f'/GLODAP/GLODAP_v2_2023_{var}_1984-2021.csv')

In [ ]:
# '''
# time indicates month as is equivalent to model's lead_time so changing it to start from 0-11 rather than 1-12
# '''


# d = csv_var(data, 'salinity')
# d['time'] = d['time'] - 1
# d



,date,year,time,day,hour,minute,lat,lon,lev,so
0,1984-07-20 14:46:00,1984.0,6.0,20.0,14.0,46.0,80.56700,7.2267,9.0,33.381000
1,1984-07-20 14:46:00,1984.0,6.0,20.0,14.0,46.0,80.56700,7.2267,9.0,33.249000
2,1984-07-20 14:46:00,1984.0,6.0,20.0,14.0,46.0,80.56700,7.2267,48.0,34.806000
3,1984-07-20 14:46:00,1984.0,6.0,20.0,14.0,46.0,80.56700,7.2267,48.0,34.803000
4,1984-07-20 14:46:00,1984.0,6.0,20.0,14.0,46.0,80.56700,7.2267,146.0,34.981000
...,...,...,...,...,...,...,...,...,...,...
1402824,2021-09-11 13:27:00,2021.0,8.0,11.0,13.0,27.0,82.45316,8.7277,1010.0,34.913117
1402825,2021-09-11 13:27:00,2021.0,8.0,11.0,13.0,27.0,82.45316,8.7277,1010.0,34.913117
1402826,2021-09-11 13:27:00,2021.0,8.0,11.0,13.0,27.0,82.45316,8.7277,1010.0,34.913117
1402827,2021-09-11 13:27:00,2021.0,8.0,11.0,13.0,27.0,82.45316,8.7277,1010.0,34.913142


: 